In [ ]:
# ¿Cómo convertir resúmenes de artículos científicos en una matriz numérica de documentos por términos?

import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.stem import PorterStemmer
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer

In [ ]:
# Se cargan los resúmenes de artículos indexados en Scopus.

abstracts = pd.read_csv("../data/scopus_abstracts.csv.gz")
abstracts.head()

In [ ]:
# ¿Cuántos documentos hay disponibles para construir la matriz?

abstracts.shape

In [ ]:
# ¿Hay resúmenes repetidos o que no aportan texto?

abstracts.abstract.value_counts().head()

In [ ]:
# Se eliminan los registros sin resumen.

abstracts = abstracts[abstracts.abstract != "[No abstract available]"]
abstracts = abstracts.reset_index(drop=True)
abstracts.shape

In [ ]:
# Un resumen es texto libre: mayúsculas, puntuación, números y, al final,
# el aviso de copyright de la editorial.

text = abstracts.abstract[6]
text

In [ ]:
# Paso 1: se quita el aviso de copyright, que empieza con el símbolo ©.
#
# Casos tipicos:
#   "... in this field. © 2023, Springer Nature."
#   "... research areas. Copyright © 2024 Inderscience Enterprises Ltd."


def remove_copyright(text):
    text = text.split("©")[0]
    text = text.replace("Copyright", "")
    return text


text = remove_copyright(text)
text[-150:]

In [ ]:
# Paso 2: se pasa el texto a minúsculas y se separa en palabras (tokens).
# Solo se conservan secuencias de letras; los números y la puntuación se descartan.


def split_into_tokens(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    return tokens


tokens = split_into_tokens(text)
len(tokens), tokens[:20]

In [ ]:
# Paso 3: se eliminan las palabras vacías (stopwords) y las palabras de menos
# de 3 letras. Aparecen en casi todos los textos y no ayudan a distinguirlos.


def remove_stopwords(tokens):
    new_tokens = []
    for token in tokens:
        if token not in ENGLISH_STOP_WORDS and len(token) > 2:
            new_tokens.append(token)
    return new_tokens


tokens = remove_stopwords(tokens)
len(tokens), tokens[:20]

In [ ]:
# Paso 4: cada palabra se reduce a su raíz (stemming), para que variantes de
# una misma palabra cuenten como un solo término.

stemmer = PorterStemmer()

stemmer.stem("study"), stemmer.stem("studies"), stemmer.stem("studied")

In [ ]:
# Se aplica el stemming a los tokens del ejemplo.


def stem_tokens(tokens):
    new_tokens = []
    for token in tokens:
        new_tokens.append(stemmer.stem(token))
    return new_tokens


tokens = stem_tokens(tokens)
len(tokens), tokens[:20]

In [ ]:
# Se aplican los cuatro pasos a todos los resúmenes. Cada resumen tokenizado
# se guarda como un texto con los tokens separados por espacios.


def tokenize(text):
    text = remove_copyright(text)
    tokens = split_into_tokens(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_tokens(tokens)
    return " ".join(tokens)


tokenized_texts = []
n_tokens = []
for text in abstracts.abstract:
    tokenized_text = tokenize(text)
    tokenized_texts.append(tokenized_text)
    n_tokens.append(len(tokenized_text.split()))

abstracts["tokens"] = tokenized_texts
abstracts["n_tokens"] = n_tokens

abstracts[["scopus_record_id", "tokens", "n_tokens"]].head()

In [ ]:
# ¿Cuántos tokens quedan en cada resumen?

plt.figure()
plt.hist(abstracts.n_tokens, bins=30, color="tab:green")
plt.xlabel("Tokens por resumen")
plt.ylabel("Número de resúmenes")
plt.show()

abstracts.n_tokens.describe()

In [ ]:
# Se construye la matriz documento-término: cada celda cuenta cuántas veces
# aparece un término en un resumen.
#
# X
# ├── filas: resúmenes (documentos)
# └── columnas: términos del vocabulario
#
# min_df=2 descarta los términos que aparecen en un solo resumen
# (nombres propios, errores de escritura, etc.).

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(abstracts.tokens)
document_term_matrix.shape

In [ ]:
# La mayoría de las celdas son cero, por eso la matriz se almacena como
# matriz dispersa: solo se guardan los valores distintos de cero.

n_cells = document_term_matrix.shape[0] * document_term_matrix.shape[1]
n_nonzero = document_term_matrix.nnz
sparsity = 1 - n_nonzero / n_cells

n_cells, n_nonzero, sparsity

In [ ]:
# Se arma el vocabulario: cada término con su columna en la matriz, el número
# de resúmenes en los que aparece y cuántas veces aparece en total.

terms = vectorizer.get_feature_names_out()
document_frequency = np.asarray((document_term_matrix > 0).sum(axis=0)).ravel()
total_count = np.asarray(document_term_matrix.sum(axis=0)).ravel()

vocabulary = pd.DataFrame(
    {
        "term_id": range(len(terms)),
        "term": terms,
        "document_frequency": document_frequency,
        "total_count": total_count,
    }
)
vocabulary.head()

In [ ]:
# ¿Cuáles son los términos más frecuentes en los resúmenes?

top_terms = vocabulary.sort_values("total_count", ascending=False).head(20)

plt.figure(figsize=(8, 6))
plt.barh(top_terms.term, top_terms.total_count, color="tab:green")
plt.gca().invert_yaxis()
plt.xlabel("Frecuencia total")
plt.show()

In [ ]:
# Se guardan la matriz, el vocabulario, los resúmenes tokenizados y la
# descripción de la matriz como evidencia verificable.

save_npz("../submission/document_term_matrix.npz", document_term_matrix)

vocabulary.to_csv("../submission/vocabulary.csv", index=False)

abstracts[["scopus_record_id", "publication_title", "tokens", "n_tokens"]].to_csv(
    "../submission/tokenized_abstracts.csv",
    index=False,
)

metadata = {
    "n_documents": document_term_matrix.shape[0],
    "n_terms": document_term_matrix.shape[1],
    "n_nonzero": n_nonzero,
    "sparsity": round(sparsity, 4),
    "min_df": 2,
    "stop_words": "sklearn ENGLISH_STOP_WORDS",
    "stemmer": "nltk PorterStemmer",
    "rows": "scopus_record_id, en el orden de tokenized_abstracts.csv",
    "columns": "term_id, en el orden de vocabulary.csv",
}

with open("../submission/matrix_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

metadata

In [ ]:
# Se comprueba que la matriz guardada se recupera igual a la original.

new_matrix = load_npz("../submission/document_term_matrix.npz")

new_matrix.shape, (new_matrix != document_term_matrix).nnz